In [1]:

# Install dependencies

!pip install streamlit pyngrok mtcnn opencv-python-headless pillow -q
print('Dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 72.8 MB/s eta 0:00:00
Dependencies installed!


In [2]:

# Mount Drive & Copy Models

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Copy models
shutil.copy('/content/drive/MyDrive/FaceAttendance/models/face_recognition.h5', 'models/')
shutil.copy('/content/drive/MyDrive/FaceAttendance/models/attention_classifier.h5', 'models/')
shutil.copy('/content/drive/MyDrive/FaceAttendance/data/label_encoder.pkl', 'models/')

print(' Models ready!')
print('People in dataset:')
import pickle
with open('models/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
print('  →', ', '.join(le.classes_))

Mounted at /content/drive
 Models ready!
People in dataset:
  → Alejandro_Toledo, Alvaro_Uribe, Amelie_Mauresmo, Andre_Agassi, Angelina_Jolie, Ariel_Sharon, Arnold_Schwarzenegger, Atal_Bihari_Vajpayee, Bill_Clinton, Carlos_Menem, Colin_Powell, David_Beckham, Donald_Rumsfeld, George_Robertson, George_W_Bush, Gerhard_Schroeder, Gloria_Macapagal_Arroyo, Gray_Davis, Guillermo_Coria, Hamid_Karzai, Hans_Blix, Hugo_Chavez, Igor_Ivanov, Jack_Straw, Jacques_Chirac, Jean_Chretien, Jennifer_Aniston, Jennifer_Capriati, Jennifer_Lopez, Jeremy_Greenstock, Jiang_Zemin, John_Ashcroft, John_Negroponte, Jose_Maria_Aznar, Juan_Carlos_Ferrero, Junichiro_Koizumi, Kofi_Annan, Laura_Bush, Lindsay_Davenport, Lleyton_Hewitt, Luiz_Inacio_Lula_da_Silva, Mahmoud_Abbas, Megawati_Sukarnoputri, Michael_Bloomberg, Naomi_Watts, Nestor_Kirchner, Paul_Bremer, Pete_Sampras, Recep_Tayyip_Erdogan, Ricardo_Lagos, Roh_Moo-hyun, Rudolph_Giuliani, Saddam_Hussein, Serena_Williams, Silvio_Berlusconi, Tiger_Woods, Tom_Daschle, To

In [3]:

# Write app.py


app_code = r'''
import streamlit as st
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle
import os
import shutil
from datetime import datetime
from mtcnn import MTCNN
from PIL import Image
import time

# ─── Page Config ───────────────────────────────────────────
st.set_page_config(
    page_title="SmartClass AI",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ─── CSS ───────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=Inter:wght@300;400;500&display=swap');

:root {
  --bg: #060910;
  --surface: #0d1117;
  --card: #111823;
  --border: #1e2d3d;
  --accent: #00e5ff;
  --accent2: #a855f7;
  --green: #22d3a5;
  --yellow: #fbbf24;
  --red: #f43f5e;
  --text: #e2e8f0;
  --muted: #4b6278;
}

html, body, [data-testid="stAppViewContainer"] {
  background-color: var(--bg) !important;
  color: var(--text) !important;
  font-family: 'Inter', sans-serif;
}
[data-testid="stSidebar"] {
  background: var(--surface) !important;
  border-right: 1px solid var(--border);
}
h1,h2,h3,h4 { font-family: 'Syne', sans-serif !important; }

.metric-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 24px; }
.metric-card {
  background: var(--card);
  border: 1px solid var(--border);
  border-radius: 14px;
  padding: 18px 20px;
  position: relative;
  overflow: hidden;
}
.metric-card::after {
  content: '';
  position: absolute; bottom: 0; left: 0; right: 0; height: 2px;
  background: linear-gradient(90deg, var(--accent), var(--accent2));
}
.metric-val { font-family: 'Syne', sans-serif; font-size: 2rem; font-weight: 800; color: var(--accent); line-height: 1; }
.metric-lbl { font-size: 0.68rem; color: var(--muted); text-transform: uppercase; letter-spacing: 2px; margin-top: 6px; }

.person-card {
  background: var(--card);
  border: 1px solid var(--border);
  border-radius: 12px;
  padding: 14px 16px;
  margin-bottom: 10px;
  transition: border-color 0.2s;
}
.person-card:hover { border-color: var(--accent); }
.person-name { font-family: 'Syne', sans-serif; font-weight: 700; font-size: 1rem; }
.person-meta { font-size: 0.72rem; color: var(--muted); margin: 4px 0 8px; }
.bar-bg { background: var(--border); border-radius: 99px; height: 6px; width: 100%; }
.bar-fill { height: 6px; border-radius: 99px; }
.badge {
  display: inline-block; padding: 2px 9px; border-radius: 20px;
  font-size: 0.65rem; font-weight: 600; letter-spacing: 0.5px;
}
.badge-green { background: rgba(34,211,165,0.12); color: #22d3a5; border: 1px solid rgba(34,211,165,0.3); }
.badge-yellow { background: rgba(251,191,36,0.12); color: #fbbf24; border: 1px solid rgba(251,191,36,0.3); }
.badge-red { background: rgba(244,63,94,0.12); color: #f43f5e; border: 1px solid rgba(244,63,94,0.3); }
.badge-blue { background: rgba(0,229,255,0.12); color: #00e5ff; border: 1px solid rgba(0,229,255,0.3); }

.section-hdr {
  font-family: 'Syne', sans-serif;
  font-size: 0.65rem; color: var(--muted);
  text-transform: uppercase; letter-spacing: 3px;
  border-bottom: 1px solid var(--border);
  padding-bottom: 8px; margin: 20px 0 14px;
}
.alert-box {
  background: rgba(244,63,94,0.08);
  border: 1px solid rgba(244,63,94,0.25);
  border-radius: 10px; padding: 12px 16px;
  margin-bottom: 8px; font-size: 0.82rem;
}
.unknown-card {
  background: rgba(168,85,247,0.08);
  border: 1px solid rgba(168,85,247,0.25);
  border-radius: 12px; padding: 14px 16px;
  margin-bottom: 10px;
}
.logo-text {
  font-family: 'Syne', sans-serif;
  font-size: 1.6rem; font-weight: 800;
  background: linear-gradient(135deg, var(--accent), var(--accent2));
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}

/* Hide camera label */
[data-testid="stCameraInput"] label { display: none; }

/* Style buttons */
.stButton > button {
  background: var(--card) !important;
  color: var(--accent) !important;
  border: 1px solid var(--border) !important;
  border-radius: 8px !important;
  font-family: 'Syne', sans-serif !important;
  font-weight: 600 !important;
  font-size: 0.8rem !important;
  letter-spacing: 0.5px !important;
  transition: all 0.2s !important;
}
.stButton > button:hover {
  border-color: var(--accent) !important;
  background: rgba(0,229,255,0.06) !important;
}

/* Dataframe */
[data-testid="stDataFrame"] { border: 1px solid var(--border) !important; border-radius: 10px; }

/* Divider */
hr { border-color: var(--border) !important; }
</style>
""", unsafe_allow_html=True)

# ─── Session State Init ────────────────────────────────────
defaults = {
    "records": [],
    "models_loaded": False,
    "person_stats": {},
    "unknown_faces": [],
    "face_model": None,
    "attn_model": None,
    "le": None,
    "detector": None,
    "session_name": "",
    "session_start": None,
    "alerts": [],
    "reg_images": [],
    "reg_name": "",
    "active_tab": "monitor",
    "frame_count": 0,
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

# ─── Helper Functions ──────────────────────────────────────
def load_models():
    try:
        fm = tf.keras.models.load_model("models/face_recognition.h5")
        am = tf.keras.models.load_model("models/attention_classifier.h5")
        with open("models/label_encoder.pkl", "rb") as f:
            le = pickle.load(f)
        det = MTCNN()
        return fm, am, le, det
    except Exception as e:
        st.error(f"Error loading models: {e}")
        return None, None, None, None

def predict_face(crop, fm, le, thr=0.60):
    img = cv2.resize(crop, (224, 224)) / 255.0
    pred = fm.predict(np.expand_dims(img, 0), verbose=0)
    conf = float(np.max(pred))
    name = le.inverse_transform([np.argmax(pred)])[0] if conf >= thr else "Unknown"
    return name, conf

def predict_attention(crop, am):
    img = cv2.resize(crop, (64, 64)) / 255.0
    score = float(am.predict(np.expand_dims(img, 0), verbose=0)[0][0])
    return round(score * 100, 1)

def draw_overlay(frame, faces_data):
    for fd in faces_data:
        x, y, w, h = fd["box"]
        is_attn = fd["attention"] >= 50
        color = (0, 229, 180) if is_attn else (244, 63, 94)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        # Tag background
        tag = f"{fd['name']}  {fd['attention']}%"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x, y - th - 14), (x + tw + 12, y), color, -1)
        cv2.putText(frame, tag, (x+6, y-6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
        # Attention bar below box
        bw = int((fd["attention"] / 100) * w)
        cv2.rectangle(frame, (x, y+h+4), (x+w, y+h+10), (30, 30, 30), -1)
        cv2.rectangle(frame, (x, y+h+4), (x+bw, y+h+10), color, -1)
    return frame

def update_stats(name, attn, conf):
    ps = st.session_state.person_stats
    now_str = datetime.now().strftime("%H:%M:%S")
    if name not in ps:
        ps[name] = {
            "first_seen": now_str,
            "last_seen": now_str,
            "attn_scores": [],
            "detections": 0,
        }
    ps[name]["last_seen"] = now_str
    ps[name]["attn_scores"].append(attn)
    ps[name]["detections"] += 1
    # Alert if distracted 3+ times in a row
    recent = ps[name]["attn_scores"][-3:]
    if len(recent) == 3 and all(s < 40 for s in recent):
        alert_msg = f"⚠️ {name} has been distracted for 3 consecutive frames!"
        if not st.session_state.alerts or st.session_state.alerts[-1] != alert_msg:
            st.session_state.alerts.append(alert_msg)
            st.session_state.alerts = st.session_state.alerts[-10:]  # keep last 10
    st.session_state.person_stats = ps

def get_summary_df():
    ps = st.session_state.person_stats
    if not ps: return pd.DataFrame()
    rows = []
    for name, d in ps.items():
        avg = round(np.mean(d["attn_scores"]), 1) if d["attn_scores"] else 0
        status = "Attentive" if avg >= 70 else ("Moderate" if avg >= 40 else "Distracted")
        rows.append({
            "Name": name,
            "Date": datetime.now().strftime("%Y-%m-%d"),
            "Session": st.session_state.session_name or "Default",
            "First Seen": d["first_seen"],
            "Last Seen": d["last_seen"],
            "Present": "Yes",
            "Avg Attention %": avg,
            "Detections": d["detections"],
            "Status": status,
        })
    return pd.DataFrame(rows)

def save_report_to_drive(df, label="summary"):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{label}_{ts}.csv"
    local = f"reports/{fname}"
    drive_path = f"/content/drive/MyDrive/FaceAttendance/reports/{fname}"
    df.to_csv(local, index=False)
    try:
        os.makedirs("/content/drive/MyDrive/FaceAttendance/reports", exist_ok=True)
        shutil.copy(local, drive_path)
        return True, drive_path
    except:
        return False, local

def people_cards_html():
    ps = st.session_state.person_stats
    if not ps:
        return "<p style='color:#4b6278;font-size:0.82rem;padding:10px 0;'>No faces detected yet…</p>"
    html = ""
    for name, d in sorted(ps.items()):
        avg = round(np.mean(d["attn_scores"]), 1) if d["attn_scores"] else 0
        if avg >= 70:
            badge = "<span class='badge badge-green'>Attentive</span>"
            bar_color = "#22d3a5"
            dot = "🟢"
        elif avg >= 40:
            badge = "<span class='badge badge-yellow'>Moderate</span>"
            bar_color = "#fbbf24"
            dot = "🟡"
        else:
            badge = "<span class='badge badge-red'>Distracted</span>"
            bar_color = "#f43f5e"
            dot = "🔴"
        html += f"""
        <div class='person-card'>
          <div style='display:flex;justify-content:space-between;align-items:center;'>
            <div class='person-name'>{dot} {name}</div>
            {badge}
          </div>
          <div class='person-meta'>First: {d['first_seen']} &nbsp;·&nbsp; Last: {d['last_seen']} &nbsp;·&nbsp; Detections: {d['detections']}</div>
          <div class='bar-bg'>
            <div class='bar-fill' style='width:{int(avg)}%;background:{bar_color};'></div>
          </div>
          <div style='font-size:0.72rem;color:#4b6278;margin-top:5px;'>Avg Attention: {avg}%</div>
        </div>"""
    return html

def unknown_cards_html():
    unk = st.session_state.unknown_faces
    if not unk:
        return "<p style='color:#4b6278;font-size:0.82rem;padding:10px 0;'>No unknown faces detected.</p>"
    html = ""
    for i, u in enumerate(unk[-5:]):
        html += f"""
        <div class='unknown-card'>
          <div style='display:flex;justify-content:space-between;'>
            <span style='font-family:Syne,sans-serif;font-weight:700;'>👤 Unknown #{i+1}</span>
            <span class='badge badge-blue'>Unregistered</span>
          </div>
          <div style='font-size:0.72rem;color:#4b6278;margin-top:4px;'>Detected at {u['time']}</div>
        </div>"""
    return html

# ─── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.markdown("<div class='logo-text'>SmartClass AI</div>", unsafe_allow_html=True)
    st.markdown("<div style='font-size:0.75rem;color:#4b6278;margin-bottom:20px;'>Real-Time Attendance & Attention</div>", unsafe_allow_html=True)

    st.markdown("<div class='section-hdr'>Session</div>", unsafe_allow_html=True)
    sess = st.text_input("Session / Period Name", value=st.session_state.session_name,
                         placeholder="e.g. CS101 — Period 2", label_visibility="collapsed")
    if sess != st.session_state.session_name:
        st.session_state.session_name = sess

    st.markdown("<div class='section-hdr'>Models</div>", unsafe_allow_html=True)
    conf_thr = st.slider("Confidence Threshold", 0.40, 0.95, 0.60, 0.05)
    attn_thr = st.slider("Attention Alert Threshold", 20, 70, 40, 5)

    if not st.session_state.models_loaded:
        if st.button("⚡ Load Models", use_container_width=True):
            with st.spinner("Loading models..."):
                fm, am, le, det = load_models()
                if fm:
                    st.session_state.face_model = fm
                    st.session_state.attn_model = am
                    st.session_state.le = le
                    st.session_state.detector = det
                    st.session_state.models_loaded = True
                    st.session_state.session_start = datetime.now().strftime("%Y-%m-%d %H:%M")
            if st.session_state.models_loaded:
                st.success(f"✅ Ready! {len(le.classes_)} people loaded")
    else:
        st.success(f"✅ Models active · {len(st.session_state.le.classes_)} people")
        if st.session_state.session_start:
            st.markdown(f"<div style='font-size:0.72rem;color:#4b6278;'>Since: {st.session_state.session_start}</div>", unsafe_allow_html=True)

    st.markdown("<div class='section-hdr'>Navigation</div>", unsafe_allow_html=True)
    tab_choice = st.radio(
        "Go to",
        [" Monitor", " Register Face", " Reports"],
        label_visibility="collapsed"
    )

    st.markdown("<div class='section-hdr'>Reset</div>", unsafe_allow_html=True)
    col_r1, col_r2 = st.columns(2)
    with col_r1:
        if st.button("🔄 New Period", use_container_width=True, help="Clear stats for new period, keep models"):
            st.session_state.records = []
            st.session_state.person_stats = {}
            st.session_state.unknown_faces = []
            st.session_state.alerts = []
            st.session_state.session_start = datetime.now().strftime("%Y-%m-%d %H:%M")
            st.rerun()
    with col_r2:
        if st.button("🗑️ Full Reset", use_container_width=True, help="Clear everything including models"):
            for k, v in defaults.items():
                st.session_state[k] = v
            st.rerun()

# ─── MAIN AREA ─────────────────────────────────────────────
active = tab_choice.split(" ", 1)[1].strip()

# ══════════════════════════════════════════════════════════════
#  TAB 1 — MONITOR
# ══════════════════════════════════════════════════════════════
if active == "Monitor":
    # Header
    hcol1, hcol2 = st.columns([3, 1])
    with hcol1:
        sess_display = st.session_state.session_name or "Default Session"
        st.markdown(f"<h2 style='margin:0;font-family:Syne,sans-serif;'> {sess_display}</h2>", unsafe_allow_html=True)
        st.markdown(f"<div style='color:#4b6278;font-size:0.8rem;'>{datetime.now().strftime('%A, %d %B %Y')}</div>", unsafe_allow_html=True)

    # ── Metrics ──
    ps = st.session_state.person_stats
    all_a = [np.mean(d["attn_scores"]) for d in ps.values() if d["attn_scores"]]
    avg_a = round(np.mean(all_a), 1) if all_a else 0
    attentive = sum(1 for d in ps.values() if d["attn_scores"] and np.mean(d["attn_scores"]) >= attn_thr)
    ac = "#22d3a5" if avg_a >= 70 else ("#fbbf24" if avg_a >= 40 else "#f43f5e")

    st.markdown(f"""
    <div class='metric-grid'>
      <div class='metric-card'>
        <div class='metric-val'>{len(ps)}</div>
        <div class='metric-lbl'>Total Detected</div>
      </div>
      <div class='metric-card'>
        <div class='metric-val' style='color:#22d3a5;'>{attentive}</div>
        <div class='metric-lbl'>Attentive</div>
      </div>
      <div class='metric-card'>
        <div class='metric-val' style='color:{ac};'>{avg_a}%</div>
        <div class='metric-lbl'>Avg Attention</div>
      </div>
      <div class='metric-card'>
        <div class='metric-val' style='color:#a855f7;'>{len(st.session_state.unknown_faces)}</div>
        <div class='metric-lbl'>Unknown Faces</div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<hr>", unsafe_allow_html=True)

    # ── Alerts ──
    if st.session_state.alerts:
        with st.expander(f"⚠️ Attention Alerts ({len(st.session_state.alerts)})", expanded=True):
            for a in reversed(st.session_state.alerts[-5:]):
                st.markdown(f"<div class='alert-box'>{a}</div>", unsafe_allow_html=True)
            if st.button("Clear Alerts"):
                st.session_state.alerts = []
                st.rerun()

    # ── Camera + Right Panel ──
    main_col, side_col = st.columns([3, 2])

    with main_col:
        st.markdown("<div class='section-hdr'>Live Camera Feed</div>", unsafe_allow_html=True)

        if not st.session_state.models_loaded:
            st.warning("⚡ Load models from the sidebar first!")
        else:
            picture = st.camera_input(" ", label_visibility="hidden")

            if picture is not None:
                img = Image.open(picture)
                frame = np.array(img)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if frame.shape[2] == 3 else frame.copy()

                with st.spinner("Analyzing…"):
                    faces = st.session_state.detector.detect_faces(frame_rgb)
                    now = datetime.now()
                    fdata = []

                    for face in faces:
                        x, y, w, h = face["box"]
                        x, y = max(0, x), max(0, y)
                        crop = frame_rgb[y:y+h, x:x+w]
                        if crop.size == 0:
                            continue
                        name, conf = predict_face(crop, st.session_state.face_model, st.session_state.le, conf_thr)
                        attn = predict_attention(crop, st.session_state.attn_model)
                        fdata.append({"box": (x, y, w, h), "name": name, "confidence": conf, "attention": attn})

                        if name != "Unknown":
                            update_stats(name, attn, conf)
                            st.session_state.records.append({
                                "Name": name,
                                "Session": st.session_state.session_name or "Default",
                                "Date": now.strftime("%Y-%m-%d"),
                                "Time": now.strftime("%H:%M:%S"),
                                "Present": "Yes",
                                "Confidence": f"{conf*100:.1f}%",
                                "Attention_%": attn,
                                "Status": "Attentive" if attn >= attn_thr else "Distracted"
                            })
                        else:
                            st.session_state.unknown_faces.append({"time": now.strftime("%H:%M:%S")})
                            st.session_state.unknown_faces = st.session_state.unknown_faces[-20:]

                    # Draw and show
                    annotated = draw_overlay(frame_rgb.copy(), fdata)
                    st.image(annotated, channels="RGB", use_container_width=True)

                    if not faces:
                        st.info("No faces detected — move closer or improve lighting.")
                    else:
                        st.success(f"✅ {len(faces)} face(s) detected!")

    with side_col:
        st.markdown("<div class='section-hdr'>Detected People</div>", unsafe_allow_html=True)
        st.markdown(people_cards_html(), unsafe_allow_html=True)

        st.markdown("<div class='section-hdr'>Unknown Faces</div>", unsafe_allow_html=True)
        st.markdown(unknown_cards_html(), unsafe_allow_html=True)
        if st.session_state.unknown_faces:
            st.caption("👉 Go to **Register Face** tab to add them.")


# ══════════════════════════════════════════════════════════════
#  TAB 2 — REGISTER FACE
# ══════════════════════════════════════════════════════════════
elif active == "Register Face":
    st.markdown("<h2 style='font-family:Syne,sans-serif;margin-bottom:4px;'>Register New Face</h2>", unsafe_allow_html=True)
    st.markdown("<div style='color:#4b6278;font-size:0.82rem;margin-bottom:20px;'>Capture images and save them to Drive. Retrain your model in Colab after registering.</div>", unsafe_allow_html=True)

    reg_col1, reg_col2 = st.columns([1, 1])

    with reg_col1:
        st.markdown("<div class='section-hdr'>Person Details</div>", unsafe_allow_html=True)
        new_name = st.text_input("Full Name", placeholder="e.g. anushka",
                                  help="Use lowercase, no spaces — matches your dataset folder style")
        target_imgs = st.slider("Target images to capture", 5, 30, 15, 5)

        if new_name:
            drive_dest = f"/content/drive/MyDrive/FaceAttendance/custom_dataset/{new_name}/"
            st.markdown(f"""
            <div style='background:#0d1117;border:1px solid #1e2d3d;border-radius:10px;padding:14px;font-size:0.78rem;'>
              <div style='color:#4b6278;margin-bottom:6px;'>Images will be saved to:</div>
              <div style='color:#00e5ff;font-family:monospace;'>{drive_dest}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("<div class='section-hdr'>Instructions</div>", unsafe_allow_html=True)
        st.markdown("""
        <div style='font-size:0.8rem;color:#4b6278;line-height:1.7;'>
        1. Enter the person's name above<br>
        2. Use camera below to capture multiple photos<br>
        3. Vary angle, lighting, expression<br>
        4. Click <b style='color:#00e5ff;'>Save All to Drive</b> when done<br>
        5. Retrain your model in Colab after saving
        </div>
        """, unsafe_allow_html=True)

    with reg_col2:
        st.markdown("<div class='section-hdr'>Capture</div>", unsafe_allow_html=True)
        n_saved = len(st.session_state.reg_images)
        progress = min(n_saved / target_imgs, 1.0)
        st.progress(progress, text=f"{n_saved}/{target_imgs} images captured")

        cam_pic = st.camera_input(" ", key="reg_cam", label_visibility="hidden")

        if cam_pic is not None and new_name:
            img = Image.open(cam_pic)
            frame = np.array(img)
            # Detect & crop face if models loaded, else save full frame
            if st.session_state.models_loaded:
                det = st.session_state.detector
                faces = det.detect_faces(frame)
                if faces:
                    x, y, w, h = faces[0]["box"]
                    x, y = max(0, x), max(0, y)
                    crop = frame[y:y+h, x:x+w]
                    if crop.size > 0:
                        st.session_state.reg_images.append(crop)
                        st.image(crop, channels="RGB", use_container_width=True,
                                 caption=f"Face #{n_saved+1} captured ✅")
                    else:
                        st.warning("Face too small — move closer.")
                else:
                    st.warning("No face detected in frame — try again.")
            else:
                st.session_state.reg_images.append(frame)
                st.image(frame, channels="RGB", use_container_width=True,
                         caption=f"Image #{n_saved+1} captured ✅")

        elif cam_pic is not None and not new_name:
            st.warning("Enter a name first!")

    # ── Save to Drive ──
    st.markdown("<hr>", unsafe_allow_html=True)
    sc1, sc2, sc3 = st.columns([2, 1, 1])
    with sc1:
        if st.session_state.reg_images:
            st.markdown(f"<div style='color:#4b6278;font-size:0.82rem;padding:8px 0;'>{len(st.session_state.reg_images)} image(s) ready to save for <b style='color:#e2e8f0;'>{new_name or '(no name)'}</b></div>", unsafe_allow_html=True)
    with sc2:
        if st.button("Save All to Drive", use_container_width=True, disabled=not (new_name and st.session_state.reg_images)):
            save_path = f"/content/drive/MyDrive/FaceAttendance/custom_dataset/{new_name}/"
            os.makedirs(save_path, exist_ok=True)
            saved = 0
            for i, img_arr in enumerate(st.session_state.reg_images):
                try:
                    ts = int(time.time() * 1000)
                    fpath = os.path.join(save_path, f"{new_name}_{ts}_{i:03d}.jpg")
                    cv2.imwrite(fpath, cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR))
                    saved += 1
                except Exception as e:
                    st.error(f"Error saving image {i}: {e}")
            st.success(f"✅ {saved} images saved to Drive for '{new_name}'!")
            st.info("📌 Remember to retrain your face_recognition model in Colab to include this person.")
            st.session_state.reg_images = []
    with sc3:
        if st.button("Clear Captures", use_container_width=True, disabled=not st.session_state.reg_images):
            st.session_state.reg_images = []
            st.rerun()


# ══════════════════════════════════════════════════════════════
#  TAB 3 — REPORTS
# ══════════════════════════════════════════════════════════════
elif active == "Reports":
    st.markdown("<h2 style='font-family:Syne,sans-serif;margin-bottom:4px;'>Attendance & Attention Reports</h2>", unsafe_allow_html=True)
    st.markdown(f"<div style='color:#4b6278;font-size:0.82rem;margin-bottom:20px;'>Session: <b style='color:#e2e8f0;'>{st.session_state.session_name or 'Default'}</b> &nbsp;·&nbsp; Started: {st.session_state.session_start or 'N/A'}</div>", unsafe_allow_html=True)

    sdf = get_summary_df()
    rdf = pd.DataFrame(st.session_state.records) if st.session_state.records else pd.DataFrame()

    # Summary stats
    if not sdf.empty:
        r1, r2, r3, r4 = st.columns(4)
        r1.metric("Total Present", len(sdf))
        r2.metric("Attentive", len(sdf[sdf["Status"] == "Attentive"]))
        r3.metric("Distracted", len(sdf[sdf["Status"] == "Distracted"]))
        r4.metric("Avg Attention", f"{sdf['Avg Attention %'].mean():.1f}%")

    st.markdown("<div class='section-hdr'>Attendance Summary</div>", unsafe_allow_html=True)
    if not sdf.empty:
        st.dataframe(sdf, use_container_width=True, hide_index=True)
    else:
        st.markdown("<p style='color:#4b6278;font-size:0.85rem;'>No data yet — run the monitor first.</p>", unsafe_allow_html=True)

    st.markdown("<div class='section-hdr'>Raw Event Log (last 50)</div>", unsafe_allow_html=True)
    if not rdf.empty:
        st.dataframe(rdf.tail(50), use_container_width=True, hide_index=True)
    else:
        st.markdown("<p style='color:#4b6278;font-size:0.85rem;'>No events logged yet.</p>", unsafe_allow_html=True)

    st.markdown("<div class='section-hdr'>Download</div>", unsafe_allow_html=True)
    dl1, dl2, dl3 = st.columns(3)
    with dl1:
        if not sdf.empty:
            st.download_button(
                "⬇️ Summary CSV",
                sdf.to_csv(index=False).encode(),
                f"summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
                "text/csv", use_container_width=True
            )
    with dl2:
        if not rdf.empty:
            st.download_button(
                "⬇️ Raw Log CSV",
                rdf.to_csv(index=False).encode(),
                f"rawlog_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
                "text/csv", use_container_width=True
            )
    with dl3:
        if not sdf.empty:
            if st.button("☁️ Save to Drive", use_container_width=True):
                ok, path = save_report_to_drive(sdf, label=st.session_state.session_name.replace(" ", "_") or "summary")
                if ok:
                    st.success(f"Saved to Drive: {path}")
                else:
                    st.warning(f"Drive save failed. Local copy at: {path}")
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written!')

✅ app.py written!


In [4]:

# Launch Dashboard via ngrok

import subprocess, threading, time
from pyngrok import ngrok, conf


NGROK_TOKEN = "3BWF8GNvc1Edk2x4zISAUk9Wv1F_4vg4PAi13bXmAH8KiZJa6"


ngrok.set_auth_token(NGROK_TOKEN)

# Kill any leftover processes
!pkill -f streamlit 2>/dev/null; sleep 1
ngrok.kill()
time.sleep(1)

def run_streamlit():
    subprocess.run([
        'python', '-m', 'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()

print("⏳ Starting Streamlit...")
time.sleep(12)

try:
    url = ngrok.connect(8501)
    print("\n" + "═"*50)
    print("✅  SmartClass AI Dashboard is LIVE!")
    print("═"*50)
    print(f"\n🌐  Open in browser:  {url}\n")
    print("═"*50)

except Exception as e:
    print(f"❌ ngrok error: {e}")
    print("Try re-running this cell or check your token.")

^C
⏳ Starting Streamlit...

══════════════════════════════════════════════════
✅  SmartClass AI Dashboard is LIVE!
══════════════════════════════════════════════════

🌐  Open in browser:  NgrokTunnel: "https://featherly-presupplemental-ladawn.ngrok-free.dev" -> "http://localhost:8501"

══════════════════════════════════════════════════
